# 전처리 2 · 직접 풀어보는 연습문제 정답지

이 노트북은 `Preprocessing_2_Feature_Scaling_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 스케일링 실습을 위한 다차원 수치 데이터를 생성합니다. 셀을 먼저 실행해주세요.
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, Normalizer

np.random.seed(42)
scaling_df = pd.DataFrame({
    "age": [22, 25, 29, 35, 38, 42, 48, 55, 60, 65],
    "income": [2800, 3200, 4100, 5000, 5600, 6200, 7100, 8500, 9200, 45000], # 45,000 고소득 이상치
    "credit_score": [650, 710, 680, 740, 800, 720, 690, 780, 820, 750]
})


## 문제 1. StandardScaler를 활용한 표준화 (Z-score 표준화) [표준화]

`scaling_df[['age', 'credit_score']]` 피처에 대해:
1. `StandardScaler` 객체를 생성하고 `.fit_transform()`으로 표준화를 적용하세요.
2. 변환된 결과의 각 컬럼별 평균(Mean)과 표준편차(Std)를 소수점 둘째 자리까지 출력하세요.
3. 변환된 2차원 배열을 데이터프레임으로 변환하여 상위 3개 행을 출력하세요.


In [ ]:
# 문제 1 풀이
features = ["age", "credit_score"]
scaler_std = StandardScaler()

scaled_std = scaler_std.fit_transform(scaling_df[features])
df_std = pd.DataFrame(scaled_std, columns=features)

print("=== StandardScaler 변환 후 상위 3행 ===")
print(df_std.head(3).round(3))

print("\n변환 후 컬럼별 평균:", df_std.mean().round(2).to_dict())
print("변환 후 컬럼별 표준편차:", df_std.std(ddof=0).round(2).to_dict())


**결과 해석 / 확인한 점:**  
1. **수학적 목표 달성:** 변환 후 평균은 0.0, 표준편차는 1.0으로 완벽히 정규화되었습니다.
2. **단위 무차원화:** 나이(세)와 신용점수(점)라는 상이한 단위가 동일한 Z-score 표준 점수 체계로 통일되어 선형 회귀나 PCA의 피처 가중치 왜곡을 방지합니다.


## 문제 2. MinMaxScaler를 활용한 0~1 구간 정규화 [정규화]

`scaling_df[['age', 'credit_score']]` 피처에 대해:
1. `MinMaxScaler` 객체를 생성하고 정규화를 적용하세요.
2. 변환 후 각 컬럼의 최솟값(Min)과 최댓값(Max)이 각각 정확히 0과 1인지 확인하세요.
3. 원래 나이가 22세였던 최소값 고객과 65세였던 최대값 고객의 변환 점수를 출력하세요.


In [ ]:
# 문제 2 풀이
scaler_minmax = MinMaxScaler()
scaled_minmax = scaler_minmax.fit_transform(scaling_df[features])
df_minmax = pd.DataFrame(scaled_minmax, columns=features)

print("=== MinMaxScaler 변환 후 통계 요약 ===")
print(df_minmax.describe().loc[["min", "max", "mean"]].round(3))

print(f"\n최연소(22세) 고객 변환값: {df_minmax.loc[0, 'age']:.2f}")
print(f"최고령(65세) 고객 변환값: {df_minmax.loc[9, 'age']:.2f}")


**결과 해석 / 확인한 점:**  
1. **고정 스케일 매핑:** 모든 값이 0.0과 1.0 사이의 닫힌 구간으로 압축되었으며, 최연소자는 0.0, 최고령자는 1.0을 부여받았습니다.
2. **주의점:** 이상치가 존재할 경우 대다수 정상 관측치가 0 근처의 좁은 구간으로 뭉개지는 한계가 있어 이상치 유무를 사전 확인해야 합니다.


## 문제 3. 이상치가 존재하는 소득(income) 피처에 대한 RobustScaler 적용 비교 [이상치 내성]

setup의 `income` 컬럼은 45,000만 원이라는 극단적 고소득 이상치를 포함하고 있습니다.
1. `StandardScaler`와 `RobustScaler`를 각각 `income`에 적용하세요.
2. 이상치 관측치(10번째 고객, 인덱스 9)에 대해 두 스케일러가 산출한 변환 값을 비교 출력하세요.
3. 대다수 정상 고객들의 스케일링 범위가 왜 RobustScaler에서 더 안정적인지 설명하세요.


In [ ]:
# 문제 3 풀이
income_data = scaling_df[["income"]]

# 1. StandardScaler vs RobustScaler
std_income = StandardScaler().fit_transform(income_data)
robust_income = RobustScaler().fit_transform(income_data)

comparison_income = pd.DataFrame({
    "원본소득": scaling_df["income"],
    "StandardScaler": std_income.flatten().round(2),
    "RobustScaler": robust_income.flatten().round(2)
})

print("=== 소득 스케일링 비교 표 ===")
print(comparison_income)

outlier_std = comparison_income.loc[9, "StandardScaler"]
outlier_rob = comparison_income.loc[9, "RobustScaler"]
print(f"\n이상치(45,000) 관측치 변환 결과 -> Standard: {outlier_std}, Robust: {outlier_rob}")


**결과 해석 / 확인한 점:**  
1. **평균 오염 방지:** `StandardScaler`는 이상치 45,000 때문에 평균이 9,990원으로 치솟아 정상 고객 9명의 점수가 모두 -0.6 ~ -0.2의 음수 구간으로 짓눌렸습니다.
2. **RobustScaler의 탁월함:** 중앙값(5,900)과 IQR(2,875)을 기준으로 분모/분자를 구성하므로 이상치에 오염되지 않고 정상 고객들의 분포를 보존했습니다.


## 문제 4. Normalizer를 활용한 행 단위(L2 노름) 샘플 정규화 [벡터 노름]

`Normalizer`는 열(피처)이 아니라 각 행(고객 샘플)의 벡터 길이를 1로 만드는 스케일러입니다.
1. `Normalizer(norm='l2')`를 사용하여 `scaling_df[['age', 'credit_score']]`를 변환하세요.
2. 변환 후 첫 번째 고객의 변환된 값들의 제곱합($x_1^2 + x_2^2$)이 1.0이 되는지 수학적으로 검증하세요.


In [ ]:
# 문제 4 풀이
features_sub = ["age", "credit_score"]
normalizer = Normalizer(norm="l2")

scaled_norm = normalizer.fit_transform(scaling_df[features_sub])
df_norm = pd.DataFrame(scaled_norm, columns=features_sub)

print("=== Normalizer(L2) 변환 결과 ===")
print(df_norm.head(3))

# 첫 번째 고객 벡터 길이 검증
c0 = df_norm.iloc[0]
norm_len = np.sqrt(c0["age"]**2 + c0["credit_score"]**2)
print(f"\n첫 번째 고객 L2 벡터 길이: {norm_len:.6f} (정확히 1.0)")


**결과 해석 / 확인한 점:**  
1. **행 단위 스케일링:** 다른 스케일러와 달리 각 관측치 벡터를 단위 구(Unit Sphere) 표면으로 투영합니다.
2. **적용 분야:** 크기(Magnitude)보다 각도나 방향성(Direction)이 중요한 코사인 유사도 기반 추천 시스템과 문서 텍스트 분류에 적합합니다.


## 문제 5. Train/Test 분할 환경에서의 올바른 스케일링 (데이터 누수 방지) [실무 원칙]

전체 10개 데이터를 7개 훈련셋(`X_train`)과 3개 테스트셋(`X_test`)으로 분할했습니다.
1. 올바른 방법: `X_train`에만 `.fit_transform()`을 수행하고, `X_test`에는 훈련셋으로 학습된 스케일러의 `.transform()`만을 적용하세요.
2. 잘못된 방법: `X_test`에 별도로 `.fit_transform()`을 적용하는 행위가 왜 데이터 누수(Data Leakage)인지 주석으로 설명하세요.


In [ ]:
# 문제 5 풀이
X_train = scaling_df.iloc[:7][["age", "income"]]
X_test = scaling_df.iloc[7:][["age", "income"]]

# 올바른 스케일러 파이프라인
scaler = StandardScaler()

# 1. 훈련셋에만 fit_transform
X_train_scaled = scaler.fit_transform(X_train)

# 2. 테스트셋에는 fit 없이 transform만 적용!
X_test_scaled = scaler.transform(X_test)

print(f"훈련셋 학습 평균 (Train Mean): age={scaler.mean_[0]:.1f}, income={scaler.mean_[1]:.1f}")
print("\n스케일링된 훈련셋 앞 3개:\n", X_train_scaled[:3].round(2))
print("\n스케일링된 테스트셋 (훈련셋 기준 적용):\n", X_test_scaled.round(2))


**결과 해석 / 확인한 점:**  
1. **엄격한 데이터 격리:** 테스트셋의 평균과 분산이 스케일러에 반영되는 순간 미래 데이터가 모델에 흘러들어가는 데이터 누수(Leakage)가 발생합니다.
2. **실무 규칙 준수:** 테스트셋은 오직 훈련셋의 모수(`scaler.mean_`, `scaler.scale_`)만을 그대로 사용하여 `.transform()`해야 실전 환경과 동일한 엄밀한 성능 평가가 가능합니다.


## 문제 6. 역변환(inverse_transform)을 통한 원래 스케일 단위 복원 [결과 복원]

스케일링된 상태로 머신러닝 모델의 예측값을 구한 후, 이를 다시 고객이 이해할 수 있는 원래 원화(원) 및 나이(세) 단위로 복원해야 합니다.
1. 문제 5에서 스케일링된 `X_test_scaled`를 `scaler.inverse_transform()`을 사용하여 역변환하세요.
2. 역변환된 결과가 원본 `X_test`와 소수점 오차 범위 내에서 일치하는지 `np.allclose()`로 검증하세요.


In [ ]:
# 문제 6 풀이
# 역변환 수행
X_test_restored = scaler.inverse_transform(X_test_scaled)
df_restored = pd.DataFrame(X_test_restored, columns=["age", "income"], index=X_test.index)

print("=== 역변환된 복원 데이터 ===")
print(df_restored.round(1))

# 원본 X_test와의 일치성 수학적 검증
is_matched = np.allclose(X_test.values, X_test_restored)
print(f"\n원본 데이터와의 완벽 일치 여부: {is_matched}")
assert is_matched, "역변환 오차 발생 오류!"


**결과 해석 / 확인한 점:**  
1. **가역성(Reversibility) 검증:** `inverse_transform()`은 스케일링 공식의 역연산($x = z \times \sigma + \mu$)을 수행하여 데이터를 100% 무손실 복원합니다.
2. **비즈니스 보고 활용:** 모델의 예측 결과를 경영진에게 보고할 때는 표준화된 Z점수가 아니라 원래 화폐 단위로 환산하여 제시해야 하므로 필수적인 기법입니다.
